# File Handling Part 2 - JSON, XML, and YAML


# Why Structured File Formats Matter
In modern data platforms, you'll handle files that aren't just rows and columns:

- APIs return JSON.
- Metadata and orchestration configs use YAML.
- Legacy systems or ERP exports still use XML.

Understanding how to read, process, and write these formats is essential to integrate sources into your data pipelines.

### 1. JSON - JavaScript Object Notation

JSON is the most common data format today.
It's lightweight, readable, and directly maps to Python dictionaries and lists.

In [0]:


import json

with open("/Volumes/thedataengineering_00/data/sampledata/sampleimages/employees.json", "r") as f:
    data = json.load(f)
    print(data)

for emp in data:
    print(emp)
    print(emp["name"], "→", emp["salary"])

json.load() parses file contents into native Python objects. If you already have JSON text in a string, use json.loads().

### Writing JSON

In [0]:
import json

employees = [
    {"id": 3, "name": "Rohan", "salary": 18000},
    {"id": 4, "name": "Neha", "salary": 17000}
]

with open("/Volumes/thedataengineering_00/data/sampledata/sampleimages/employees_out.json", "w") as f:
    json.dump(employees, f, indent=4)

print("JSON file written successfully!")


with open("/Volumes/thedataengineering_00/data/sampledata/sampleimages/employees_out.json", "r") as f:
    data = json.load(f)
    print(data)

### Great for incremental ingestion or metadata appending.

In [0]:
with open("/Volumes/thedataengineering_00/data/sampledata/sampleimages/employees.json", "r") as f:
    data = json.load(f)

data.append({"id": 5, "name": "Raj", "salary": 16000})

with open("/Volumes/thedataengineering_00/data/sampledata/sampleimages/employees.json", "w") as f:
    json.dump(data, f, indent=4)

In [0]:
import json

with open("/Volumes/thedataengineering_00/data/sampledata/sampleimages/employees.json", "r") as f:
    data = json.load(f)

for emp in data:
    print(emp["name"], "→", emp["salary"])

# 2. XML - eXtensible Markup Language
XML is common in enterprise systems (ERP, banking, SAP).
 Though verbose, it's still widely used for configuration and legacy integrations.

In [0]:
import xml.etree.ElementTree as ET

tree = ET.parse("/Volumes/thedataengineering_00/data/sampledata/sampleimages/employees.xml")
root = tree.getroot()

for emp in root.findall("employee"):
    name = emp.find("name").text
    salary = emp.find("salary").text
    print(name, "→", salary)

### Writing XML

In [0]:
import xml.etree.ElementTree as ET

root = ET.Element("employees")

e1 = ET.SubElement(root, "employee")
ET.SubElement(e1, "id").text = "3"
ET.SubElement(e1, "name").text = "Rohan"
ET.SubElement(e1, "salary").text = "18000"

tree = ET.ElementTree(root)
tree.write("/Volumes/thedataengineering_00/data/sampledata/sampleimages/employees_new.xml")

Produces a valid XML file - useful when exporting processed data to systems expecting XML.

### Converting XML to Dictionary

Often, we want to work with XML as a Python dict.
 For this, use a small helper:

In [0]:
def xml_to_dict(elem):
    return {child.tag: child.text for child in elem}

tree = ET.parse("/Volumes/thedataengineering_00/data/sampledata/sampleimages/employees.xml")
root = tree.getroot()
records = [xml_to_dict(e) for e in root.findall("employee")]
print(records)

### YAML - Yet Another Markup Language

YAML is popular for configuration and metadata in Airflow, Databricks, Docker, and MLflow.
It's like JSON, but more human-friendly.

### Reading YAML

In [0]:
import yaml

with open("/Volumes/thedataengineering_00/data/sampledata/sampleimages/employees.yaml", "r") as f:
    config = yaml.safe_load(f)

print(config["pipeline"]["name"])
print(config["pipeline"]["transformations"])

### Writing YAML

In [0]:
import yaml

data = {
    "pipeline": {
        "name": "customer_etl",
        "source": "customers_raw.csv",
        "destination": "customers_clean.csv"
    }
}

with open("/Volumes/thedataengineering_00/data/sampledata/sampleimages/config.yaml", "w") as f:
    yaml.dump(data, f)

### Mini Data Engineering Scenario

Let's connect it all - read a YAML config, load JSON data, and write a CSV.

In [0]:
import json, yaml, csv

# Load config
with open("/Volumes/thedataengineering_00/data/sampledata/sampleimages/employees_config.yaml", "r") as f:
    config = yaml.safe_load(f)

# Read JSON
with open(config["source_file"], "r") as f:  /Volumes/thedataengineering_00/data/sampledata/sampleimages/employees.json
    data = json.load(f)

# Clean and write CSV
with open(config["target_file"], "w", newline="") as f:  /Volumes/thedataengineering_00/data/sampledata/sampleimages/employees_clean.csv
    writer = csv.DictWriter(f, fieldnames=config["fields"])
    writer.writeheader()
    for record in data:
        record["name"] = record["name"].strip().title()
        writer.writerow(record)

print("Cleaned CSV written successfully using YAML config!")

### Congratulations - you just built a multi-format ETL pipeline using YAML for configuration, JSON as input, and CSV as output.

Practice Tasks

- Read a JSON file and write it as CSV.
- Convert XML employee data to a list of Python dictionaries.
- Write a YAML config to define file paths and transformations.
- Build a mini ETL using the YAML + JSON example above.
- Modify the YAML config to change target file and rerun pipeline.